# 4.2. Metrics: Similarity (Word Overlap, Cosine Similarity, BLEU)

In [1]:
import pandas as pd

In [2]:
# Reading the data from the file
df = pd.read_csv("../data/raw/paranmt_for_detox_500k.tsv", sep="\t", index_col=0)
df.head()

,reference,translation,similarity,length_diff,ref_tox,trn_tox
0,"If Alkar is flooding her with psychic waste, t...","if Alkar floods her with her mental waste, it ...",0.785171,0.010309,0.014195,0.981983
1,Now you're getting nasty.,you're becoming disgusting.,0.749687,0.071429,0.065473,0.999039
2,"Well, we could spare your life, for one.","well, we can spare your life.",0.919051,0.268293,0.213313,0.985068
3,"Ah! Monkey, you've got to snap out of it.","monkey, you have to wake up.",0.664333,0.309524,0.053362,0.994215
4,I've got orders to put her down.,I have orders to kill her.,0.726639,0.181818,0.009402,0.999348


## 1. Word Overlap

In [3]:
import string


def get_wo_score(reference, hypothesis):
    # Remove all punctuation and replace with whitespace
    reference = reference.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))
    hypothesis = hypothesis.translate(str.maketrans(string.punctuation, " " * len(string.punctuation)))

    # Calculate the number of words in the reference and hypothesis
    ref_words = set(reference.lower().split())
    hyp_words = set(hypothesis.lower().split())

    inter_len = len(ref_words.intersection(hyp_words))
    union_len = len(ref_words.union(hyp_words))

    if union_len == 0:
        return 0

    return inter_len / union_len

## 2. Cosine Similarity

In [4]:
import spacy

nlp = spacy.load("en_core_web_lg")

def get_cosine_score(reference, hypothesis):
    ref_doc = nlp(reference)
    hyp_doc = nlp(hypothesis)

    return ref_doc.similarity(hyp_doc)

## 3. BLEU

In [7]:
import nltk

def get_bleu_score(reference, hypothesis):
    return nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)

[nltk_data] Error loading en: Package 'en' not found in index


In [6]:
# Compare some random sentences from the dataset
for i in range(10):
    # Get a random index
    index = df.sample().index[0]

    # Get the reference and hypothesis sentences
    reference = df.loc[index, "reference"]
    hypothesis = df.loc[index, "translation"]

    # Calculate scores
    wo_score = get_wo_score(reference, hypothesis)
    cs_score = get_cosine_score(reference, hypothesis)
    bleu_score = get_bleu_score(reference, hypothesis)

    # Print the results
    print("Reference:   ", reference)
    print("Hypothesis:  ", hypothesis)
    print("WO score:    ", wo_score)
    print("Cosine score:", cs_score)
    print("BLEU score:  ", bleu_score)
    print()

Reference:    What the fuck am I supposed to do?
Hypothesis:   what am I supposed to do?
WO score:     0.75
Cosine score: 0.9806725687866961
BLEU score:   0.6438823761211517

Reference:    Pathetic! Of all the objections to warfare, it's the use of sunglasses!
Hypothesis:   of all objections to warfare, you choose wearing sunglasses!
WO score:     0.42857142857142855
Cosine score: 0.9197379888249169
BLEU score:   0.5790157267006715

Reference:    You make it sound like I'm already dead.
Hypothesis:   you act like I'm already dead.
WO score:     0.6
Cosine score: 0.9508865911050903
BLEU score:   0.5927377534581625

Reference:    Guys don't talk about shit like that.
Hypothesis:   guys don't talk about it like that.
WO score:     0.7777777777777778
Cosine score: 0.9659208206535234
BLEU score:   0.8721827599607718

Reference:    It's always drugs with these Dowling guys.
Hypothesis:   with these Dowling men, they're always drugs.
WO score:     0.45454545454545453
Cosine score: 0.830139253